In [ ]:
!pip install docent-python

In [3]:
from datasets import load_dataset

# Load the Verified dataset
# Note: Verified usually uses the 'test' split for evaluation
dataset = load_dataset("SWE-bench/SWE-bench_Verified", split="test")

# Convert to a dictionary for easy lookup by instance_id
ground_truth = {ins["instance_id"]: ins["patch"] for ins in dataset}


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [4]:
HARD_PROBLEMS = [
    # Pattern 1: Infinite loop / churn spiral (7) — hit $3 cost ceiling, never converged
    "django__django-10554",
    "django__django-11265",
    "django__django-11734",
    "django__django-13344",
    "django__django-15554",
    "django__django-16263",
    "sphinx-doc__sphinx-9229",

    # Pattern 2: Pylint blind spot (8) — 80% fail rate, lacks AST-visitor architecture insight
    "pylint-dev__pylint-4551",
    "pylint-dev__pylint-4604",
    "pylint-dev__pylint-4661",
    "pylint-dev__pylint-6386",
    "pylint-dev__pylint-6528",
    "pylint-dev__pylint-7080",
    "pylint-dev__pylint-7277",
    "pylint-dev__pylint-8898",

    # Pattern 3: Sphinx build maze (11) — lost in deeply-coupled extension system, avg cost $1.80
    "sphinx-doc__sphinx-10435",
    "sphinx-doc__sphinx-10614",
    "sphinx-doc__sphinx-11510",
    "sphinx-doc__sphinx-7462",
    "sphinx-doc__sphinx-7590",
    "sphinx-doc__sphinx-7748",
    "sphinx-doc__sphinx-7985",
    "sphinx-doc__sphinx-8548",
    "sphinx-doc__sphinx-9461",
    "sphinx-doc__sphinx-9602",

    # Pattern 4: Overconfident quick fail (5 unique) — <20 API calls, skipped verification
    "django__django-10999",
    "django__django-14140",
    "django__django-14315",
    # (pylint-4661 and pylint-7277 also here but listed in Pattern 2)

    # Pattern 5: Sympy math reasoning gap (21) — lacks domain-specific symbolic math insight
    "sympy__sympy-13031",
    "sympy__sympy-13798",
    "sympy__sympy-13852",
    "sympy__sympy-13877",
    "sympy__sympy-13974",
    "sympy__sympy-14248",
    "sympy__sympy-14976",
    "sympy__sympy-15599",
    "sympy__sympy-15875",
    "sympy__sympy-16597",
    "sympy__sympy-17318",
    "sympy__sympy-17630",
    "sympy__sympy-18199",
    "sympy__sympy-18763",
    "sympy__sympy-19495",
    "sympy__sympy-20428",
    "sympy__sympy-20438",
    "sympy__sympy-21596",
    "sympy__sympy-21930",
    "sympy__sympy-22080",
    "sympy__sympy-23950",

    # Pattern 6: High-churn wrong answer (10 unique) — >$1.50 cost, submitted wrong, tunnel-visioned
    "astropy__astropy-13398",
    "astropy__astropy-14598",
    "django__django-12663",
    "django__django-14034",
    "django__django-15957",
    "django__django-15973",
    "matplotlib__matplotlib-25311",
    "matplotlib__matplotlib-26208",
    "pydata__xarray-6599",
    "pydata__xarray-7229",
]

In [5]:
import pandas as pd
csv_path = "/content/agent-runs-b038912e-0133-4594-b093-92806f8ffb17 (1).csv" # replace with desired model run

df = pd.read_csv(csv_path)
run_id_map = df.set_index("metadata.instance_id")["agent_run_id"].to_dict()



In [7]:
from docent import Docent
from google.colab import userdata
import os

# Initialize client - Note: You will need to provide a valid API key
# In a real scenario, use userdata.get('DOCENT_API_KEY')
DOCENT_API_KEY = userdata.get('DOCENT_API_KEY')
client = Docent(api_key=DOCENT_API_KEY)
COLLECTION_ID = "b038912e-0133-4594-b093-92806f8ffb17"

def get_model_attempt(instance_id):
    """Returns the model's full attempt transcript as a string for a given instance_id."""
    agent_run_id = run_id_map.get(instance_id)
    if not agent_run_id:
        return None

    try:
        # 1. Get the run metadata to find transcript IDs
        run = client.get_agent_run(COLLECTION_ID, agent_run_id)

        # 2. Fetch the actual transcript content
        transcript_id = run.transcripts[0].id
        transcript = client.get_transcript(COLLECTION_ID, transcript_id)

        output = [f"## Instance: {instance_id} (Run: {agent_run_id})\n"]

        # 3. Iterate through events (Steps)
        for i, event in enumerate(transcript.events):
            output.append(f"### STEP {i+1}")
            if hasattr(event, 'thought') and event.thought:
                output.append(f"**THOUGHT:** {event.thought}")
            if hasattr(event, 'action') and event.action:
                output.append(f"**ACTION:**\n```\n{event.action}\n```")
            if hasattr(event, 'observation') and event.observation:
                output.append(f"**OBSERVATION:**\n```\n{event.observation[:1000]}\n```") # Truncated for readability
            output.append("\n---\n")

        return "\n".join(output)
    except Exception as e:
        return f"Error retrieving transcript for {instance_id}: {str(e)}"

16:40:00 [INFO] docent.sdk._base: Authenticating Docent client with frontend_url='https://docent.transluce.org' and api_url='https://api.docent.transluce.org/rest'


INFO:docent.sdk._base:Authenticating Docent client with frontend_url='https://docent.transluce.org' and api_url='https://api.docent.transluce.org/rest'


16:40:00 [INFO] docent.sdk._base: Logged in with API key


INFO:docent.sdk._base:Logged in with API key


In [10]:
missing_instance_ids = []

def write_comparison_file():
    filename = f"{COLLECTION_ID}_comparison.md"
    with open(filename, "w") as f:
        f.write(f"# Comparison Report for Collection {COLLECTION_ID}\n\n")

        for instance_id in HARD_PROBLEMS:
            # Check if we have both ground truth and a run_id
            has_gt = instance_id in ground_truth
            has_run = instance_id in run_id_map

            if not has_gt or not has_run:
                missing_instance_ids.append(instance_id)
                continue

            f.write(f"# INSTANCE ID: {instance_id}\n\n")

            # Write Model Attempt
            f.write("## MODEL ATTEMPT\n")
            attempt_text = get_model_attempt(instance_id)
            if attempt_text:
                f.write(attempt_text + "\n")
            else:
                f.write("Failed to retrieve attempt.\n")

            # Write Ground Truth
            f.write("\n## GROUND TRUTH PATCH\n")
            f.write(f"```diff\n{ground_truth[instance_id]}\n```\n")
            f.write("\n" + "="*80 + "\n\n")

    print(f"Comparison file written to: {filename}")
    if missing_instance_ids:
        print(f"Missing data for: {missing_instance_ids}")

write_comparison_file()

Comparison file written to: b038912e-0133-4594-b093-92806f8ffb17_comparison.md
